# GherProof Siamese flood-change baseline (Colab)

This notebook downloads the public S1GFloods archive and prepares an event-level Siamese U-Net experiment. It refuses to train until a curator supplies source-backed `event_id` and official or independently documented event split assignments. The downloaded archive in this project has generic image names and no such metadata; **do not fabricate event groups or use a random patch split**. This notebook is a gated recipe, not evidence that a model was trained.

The authors describe S1GFloods as research/non-commercial. The Hugging Face card and author README have conflicting license statements. Use only for non-commercial research pending clarification and preserve attribution. Benchmark metrics will not establish Sujatpur accuracy.

In [ ]:
# Colab: select a GPU runtime, then verify actual runtime availability.
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before this training workflow.')
print(torch.cuda.get_device_name(0))

In [ ]:
# Download the exact public Hugging Face RAR and verify the published LFS SHA-256.
from pathlib import Path
import hashlib, requests, subprocess
archive = Path('/content/S1GFloods.rar')
url = 'https://huggingface.co/datasets/Tamer-Saleh/S1GFloods/resolve/main/S1GFloods.rar'
expected_size = 673724972
expected_sha256 = '4f1b57c342c661853dec0d7384709ce4bb9e53097ab31cd060cd4a3f915e3e67'
if not archive.exists():
    with requests.get(url, stream=True, timeout=(30, 180)) as response:
        response.raise_for_status()
        with archive.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk: f.write(chunk)
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
assert archive.stat().st_size == expected_size, archive.stat().st_size
assert digest == expected_sha256, digest
print('Verified bytes:', archive.stat().st_size, 'SHA-256:', digest)

In [ ]:
# RAR5 extraction requires the unrar executable.
!apt-get -qq update && apt-get -qq install -y unrar
root = Path('/content/S1GFloods')
root.mkdir(exist_ok=True)
subprocess.run(['unrar', 'x', '-o+', str(archive), str(root) + '/'], check=True)
for folder in ['A', 'B', 'Label']:
    files = sorted((root / folder).glob('*.png'))
    print(folder, len(files))
    assert len(files) == 5360
print('Archive provides image folders only; no event/split manifest is presumed.')

## Supply a defensible event manifest before training

Create/upload `/content/split_manifest.csv` with one row per pair and columns `filename,event_id,split`; `filename` is e.g. `image_1.png`, `event_id` must come from source-backed dataset metadata, and `split` must be `train`, `validation`, or `test`. All files for an event must stay in one split. The current dataset archive does not contain `event_id`; obtain this mapping from the authors or documented source annotations. The following cell intentionally raises until the event-level grouping is supplied.

In [ ]:
import pandas as pd
manifest_path = Path('/content/split_manifest.csv')
if not manifest_path.exists():
    raise FileNotFoundError('Training blocked: provide a source-backed event-level split_manifest.csv; random patch splits are prohibited.')
manifest = pd.read_csv(manifest_path)
required = {'filename', 'event_id', 'split'}
assert required.issubset(manifest.columns), required - set(manifest.columns)
assert manifest['filename'].is_unique and manifest['event_id'].notna().all()
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert (manifest.groupby('event_id')['split'].nunique() == 1).all(), 'Event leakage across splits'
for row in manifest.itertuples():
    assert (root/'A'/row.filename).is_file() and (root/'B'/row.filename).is_file() and (root/'Label'/row.filename).is_file()
print(manifest.groupby('split').agg(pairs=('filename','count'), events=('event_id','nunique')))

In [ ]:
# Siamese U-Net baseline with shared encoder and temporal feature fusion.
import numpy as np, PIL.Image as Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
class PairDataset(Dataset):
    def __init__(self, rows): self.rows = rows.reset_index(drop=True)
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        name = self.rows.iloc[i].filename
        a = np.asarray(Image.open(root/'A'/name).convert('RGB'), dtype=np.float32)/255.0
        b = np.asarray(Image.open(root/'B'/name).convert('RGB'), dtype=np.float32)/255.0
        y = np.asarray(Image.open(root/'Label'/name).convert('L'))
        return torch.from_numpy(a).permute(2,0,1), torch.from_numpy(b).permute(2,0,1), torch.from_numpy((y==255).astype(np.float32))[None]
class Block(nn.Module):
    def __init__(self, a,b):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)
class SiameseUNet(nn.Module):
    def __init__(self):
        super().__init__(); self.e1=Block(3,32); self.e2=Block(32,64); self.e3=Block(64,128); self.pool=nn.MaxPool2d(2); self.up2=nn.ConvTranspose2d(256,64,2,2); self.d2=Block(128,64); self.up1=nn.ConvTranspose2d(128,32,2,2); self.d1=Block(64,32); self.out=nn.Conv2d(32,1,1)
    def forward(self,a,b):
        a1=self.e1(a); b1=self.e1(b); a2=self.e2(self.pool(a1)); b2=self.e2(self.pool(b1)); a3=self.e3(self.pool(a2)); b3=self.e3(self.pool(b2)); z=torch.cat([a3,b3],1); z=self.d2(torch.cat([self.up2(z),torch.abs(a2-b2)],1)); z=self.d1(torch.cat([self.up1(torch.cat([z,torch.abs(a2-b2)],1)),torch.abs(a1-b1)],1)); return self.out(z)
# Note: verify/tune architecture and input convention against author code before scientific use.

In [ ]:
# Event-held-out train/validation/test loop; test is evaluated once at the end.
from sklearn.metrics import confusion_matrix, precision_recall_curve, auc, average_precision_score
device=torch.device('cuda'); batch_size=8; epochs=30; seed=42
torch.manual_seed(seed); np.random.seed(seed)
loaders={s:DataLoader(PairDataset(manifest[manifest.split==s]),batch_size=batch_size,shuffle=(s=='train'),num_workers=2,pin_memory=True) for s in ['train','validation','test']}
model=SiameseUNet().to(device); opt=torch.optim.AdamW(model.parameters(),lr=1e-4); loss_fn=nn.BCEWithLogitsLoss(); scaler=torch.amp.GradScaler('cuda')
def evaluate(loader):
    model.eval(); ys=[]; scores=[]
    with torch.no_grad():
        for a,b,y in loader:
            a,b,y=a.to(device),b.to(device),y.to(device)
            with torch.autocast('cuda',dtype=torch.float16): logits=model(a,b)
            scores.append(torch.sigmoid(logits).float().cpu().numpy().ravel()); ys.append(y.cpu().numpy().ravel())
    ys=np.concatenate(ys); scores=np.concatenate(scores); ps=(scores>0.5).astype(np.uint8)
    tn,fp,fn,tp=confusion_matrix(ys,ps,labels=[0,1]).ravel(); precision=tp/(tp+fp+1e-9); recall=tp/(tp+fn+1e-9); f1=2*precision*recall/(precision+recall+1e-9); iou=tp/(tp+fp+fn+1e-9); pr,rc,_=precision_recall_curve(ys,scores)
    return {'precision':float(precision),'recall':float(recall),'f1':float(f1),'iou':float(iou),'pr_auc':float(auc(rc,pr)),'average_precision':float(average_precision_score(ys,scores)),'confusion_matrix':[[int(tn),int(fp)],[int(fn),int(tp)]]}
best=-1; patience=5; stale=0; history=[]
for epoch in range(epochs):
    model.train(); losses=[]
    for a,b,y in loaders['train']:
        a,b,y=a.to(device),b.to(device),y.to(device); opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16): logits=model(a,b); loss=loss_fn(logits,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(float(loss.detach()))
    val=evaluate(loaders['validation']); history.append({'epoch':epoch+1,'train_loss':float(np.mean(losses)),**val}); print(history[-1])
    if val['f1']>best+1e-4: best=val['f1']; stale=0; torch.save({'state_dict':model.state_dict(),'epoch':epoch+1,'seed':seed,'validation':val},'/content/gherproof_siamese_unet.pt')
    else: stale+=1
    if stale>=patience: print('Early stopping'); break
checkpoint=torch.load('/content/gherproof_siamese_unet.pt',map_location=device); model.load_state_dict(checkpoint['state_dict']); test_metrics=evaluate(loaders['test']); print('HELD-OUT EVENT TEST:',test_metrics)
# Export checkpoint, event split manifest, package versions and this metrics JSON together. Benchmark scores do not imply Sujatpur accuracy.

In [ ]:
# Patch-level inference helper. Output is a benchmark PNG mask, not a georeferenced Sujatpur result.
from PIL import Image
def predict_pair(pre_path, post_path, checkpoint_path='/content/gherproof_siamese_unet.pt', output_path='/content/predicted_change_mask.png'):
    ckpt=torch.load(checkpoint_path,map_location=device); model.load_state_dict(ckpt['state_dict']); model.eval()
    def load(path):
        x=np.asarray(Image.open(path).convert('RGB'),dtype=np.float32)/255.0
        return torch.from_numpy(x).permute(2,0,1)[None].to(device)
    a,b=load(pre_path),load(post_path)
    with torch.no_grad(), torch.autocast('cuda',dtype=torch.float16): prob=torch.sigmoid(model(a,b))[0,0].float().cpu().numpy()
    mask=(prob>0.5).astype(np.uint8)*255
    Image.fromarray(mask,mode='L').save(output_path)
    return {'output':output_path,'threshold':0.5,'positive_fraction':float(mask.mean()/255),'georeferenced':False,'use':'benchmark PNG pairs only'}
# Example after a successful, valid event-split training run:
# predict_pair(root/'A'/'image_1.png', root/'B'/'image_1.png')